In [2]:
# =================================================================================================
# NHANES 2007-2008 DIETARY MERGE - FIDELITY PRESERVING
#
# Source files:
#   DR1TOT_E_corrected.csv
#   DR2TOT_E_corrected.csv
#
# Purpose:
#   Merge Day 1 + Day 2 Total Nutrient Intake files into one participant-level Dietary CSV.
#
# Final structure:
#
#   SEQN
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#   [all Day 1 variables]
#   [all Day 2 variables]
#
# Fidelity rules:
#
#   - Preserve Dietary decimal text AS IS
#   - Do NOT round
#   - Do NOT use float_format=
#   - Keep one SEQN
#   - Keep shared variables once
#   - Validate shared variables before deduplicating
#   - Prevent engineered fields such as:
#
#       WTDRD1_DAY_2_TOTAL_INTAKE
#       WTDR2D_DAY_2_TOTAL_INTAKE
#       DRABF_DAY_2_TOTAL_INTAKE
#       DRDINT_DAY_2_TOTAL_INTAKE
#
#   - Every participant-level merge must be one-to-one
#
# CDC 2007-2008:
#
#   Total rows in DR1TOT_E / DR2TOT_E = 9,762
#   Day 1 complete respondents          = 9,255
#   Day 2 complete respondents          = 7,838
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_E_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_E_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2007_2008"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2007_2008_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# VERY IMPORTANT:
#
# dtype=str preserves corrected Dietary decimal representation.
#
# Example:
#
#     0.028999999999
#
# stays:
#
#     0.028999999999
#
# We are merging, not recalculating nutrient values.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 110)
print("NHANES 2007-2008 DIETARY DAY 1 + DAY 2 MERGE")
print("=" * 110)


print(
    "DR1TOT_E shape:",
    day1.shape
)


print(
    "DR2TOT_E shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. CYCLE-SPECIFIC EXPECTED COUNTS
# =================================================================================================

EXPECTED_ROWS = 9762

EXPECTED_DAY1_WEIGHT_NONMISSING = 9255

EXPECTED_DAY2_WEIGHT_NONMISSING = 7838


# =================================================================================================
# 6. VERIFY ROW COUNTS
# =================================================================================================

if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        f"Unexpected DR1TOT_E row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        f"Unexpected DR2TOT_E row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


print(
    "PASS: Both Dietary files contain 9,762 rows."
)


# =================================================================================================
# 7. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(df, name):

    if "SEQN" not in df.columns:

        raise ValueError(
            f"{name}: SEQN is missing."
        )


    blank_count = int(
        df["SEQN"]
        .eq("")
        .sum()
    )


    duplicate_count = int(
        df["SEQN"]
        .duplicated()
        .sum()
    )


    dot_zero_count = int(
        (
            df["SEQN"].ne("")
            &
            df["SEQN"]
            .str.endswith(".0")
        )
        .sum()
    )


    invalid_count = int(
        (
            df["SEQN"].ne("")
            &
            ~df["SEQN"]
            .str.fullmatch(r"\d+")
        )
        .sum()
    )


    print(
        f"\n{name}:"
    )


    print(
        "  Rows:",
        f"{len(df):,}"
    )


    print(
        "  Unique SEQN:",
        f"{df['SEQN'].nunique():,}"
    )


    print(
        "  Blank SEQN:",
        blank_count
    )


    print(
        "  Duplicate SEQN:",
        duplicate_count
    )


    print(
        "  SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "  Invalid SEQN formatting:",
        invalid_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: SEQN still contains '.0'."
        )


    if invalid_count != 0:

        raise ValueError(
            f"{name}: invalid SEQN formatting."
        )


# =================================================================================================
# 8. VALIDATE SEQN
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_E"
)


validate_seqn(
    day2,
    "DR2TOT_E"
)


print(
    "\nPASS: SEQN integrity validated."
)


# =================================================================================================
# 9. VERIFY SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET CHECK ---"
)


print(
    "Only in DR1TOT_E:",
    len(only_day1)
)


print(
    "Only in DR2TOT_E:",
    len(only_day2)
)


if only_day1 or only_day2:

    raise ValueError(
        "DR1TOT_E and DR2TOT_E do not contain "
        "the same participant universe."
    )


print(
    "PASS: Both files contain the same 9,762 SEQN."
)


# =================================================================================================
# 10. IDENTIFY SHARED COLUMN NAMES
# =================================================================================================

shared_detected = [

    col

    for col in day1.columns

    if (
        col in day2.columns
        and
        col != "SEQN"
    )

]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)


print(
    shared_detected
)


# =================================================================================================
# 11. EXPECTED OFFICIAL SHARED VARIABLES
# =================================================================================================

#
# CDC explicitly documents that these variable names
# are shared between Day 1 and Day 2 files.
#

SHARED_COLUMNS = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


if set(shared_detected) != set(SHARED_COLUMNS):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_detected}"
    )


print(
    "PASS: Exactly four expected shared variables found."
)


# =================================================================================================
# 12. VALIDATE SHARED VARIABLES
# =================================================================================================

#
# Compare numerically to avoid false differences such as:
#
#     "0"
# versus
#     "0.0"
#
# Original source text is NOT changed.
#

print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


shared_comparison = (

    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="outer",

        suffixes=(
            "_DAY1",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


for col in SHARED_COLUMNS:

    left_raw = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    right_raw = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    left_missing = (
        left_raw.eq("")
    )


    right_missing = (
        right_raw.eq("")
    )


    missing_mismatch = int(

        (
            left_missing
            !=
            right_missing
        )

        .sum()

    )


    both_populated = (
        ~left_missing
        &
        ~right_missing
    )


    left_numeric = pd.to_numeric(

        left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch = int(
        (~numeric_equal)
        .sum()
    )


    text_difference = int(

        (
            left_raw.loc[
                both_populated
            ]
            .reset_index(
                drop=True
            )

            !=

            right_raw.loc[
                both_populated
            ]
            .reset_index(
                drop=True
            )
        )

        .sum()

    )


    print(
        f"\n{col}"
    )


    print(
        "  Missing-position mismatches:",
        missing_mismatch
    )


    print(
        "  Numeric mismatches:",
        numeric_mismatch
    )


    print(
        "  Text-format differences:",
        text_difference
    )


    if missing_mismatch != 0:

        raise ValueError(
            f"{col}: missing positions differ "
            "between Day 1 and Day 2."
        )


    if numeric_mismatch != 0:

        raise ValueError(
            f"{col}: true numeric values differ "
            "between Day 1 and Day 2."
        )


print(
    "\nPASS: All four shared variables "
    "match between DR1TOT_E and DR2TOT_E."
)


# =================================================================================================
# 13. VALIDATE OFFICIAL DIETARY SAMPLE WEIGHT COUNTS
# =================================================================================================

#
# CDC:
#
# WTDRD1 = 9,255 populated
# WTDR2D = 7,838 populated
#

day1_weight_count = int(
    day1["WTDRD1"]
    .ne("")
    .sum()
)


day2_weight_count = int(
    day1["WTDR2D"]
    .ne("")
    .sum()
)


print(
    "\n--- CDC SAMPLE WEIGHT CHECK ---"
)


print(
    "WTDRD1 populated:",
    f"{day1_weight_count:,}"
)


print(
    "WTDR2D populated:",
    f"{day2_weight_count:,}"
)


if (
    day1_weight_count
    !=
    EXPECTED_DAY1_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDRD1 populated count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY1_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day1_weight_count:,}"
    )


if (
    day2_weight_count
    !=
    EXPECTED_DAY2_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDR2D populated count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY2_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day2_weight_count:,}"
    )


print(
    "PASS: Dietary weight sample sizes match CDC."
)


# =================================================================================================
# 14. VALIDATE DRDINT
# =================================================================================================

#
# CDC 2007-2008:
#
# 1 = Day 1 only       1,417
# 2 = Day 1 and Day 2  7,838
# missing                507
#

drdint_counts = (
    day1["DRDINT"]
    .value_counts(
        dropna=False
    )
    .to_dict()
)


print(
    "\n--- DRDINT CHECK ---"
)


print(
    drdint_counts
)


EXPECTED_DRDINT = {

    "1": 1417,

    "2": 7838,

    "": 507

}


if drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint_counts}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 15. SAVE ORIGINAL COLUMN ORDER
# =================================================================================================

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 16. REMOVE ONLY REDUNDANT DAY 2 COPIES
# =================================================================================================

#
# Canonical copies are retained from DR1TOT_E.
#
# This prevents engineered duplicate fields.
#

day2_for_merge = (

    day2.drop(
        columns=SHARED_COLUMNS
    )

)


print(
    "\nRemoved redundant DR2TOT_E copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 17. VERIFY NO OTHER COLUMN COLLISIONS
# =================================================================================================

remaining_overlap = sorted(

    (
        set(day1.columns)
        &
        set(day2_for_merge.columns)
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected overlapping columns remain before merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected column collisions remain."
)


# =================================================================================================
# 18. MERGE DAY 1 + DAY 2
# =================================================================================================

dietary = (

    day1.merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

)


#
# Outer merge may create NaN for absent values.
# Restore CSV-style blanks.
#

dietary = (
    dietary.fillna("")
)


print(
    "\nPASS: Day 1 + Day 2 merge completed."
)


# =================================================================================================
# 19. FINAL ROW / SEQN CHECK
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if dietary["SEQN"].nunique() != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary unique SEQN count changed."
    )


if dietary["SEQN"].duplicated().any():

    raise ValueError(
        "Merged Dietary file contains duplicate SEQN."
    )


if dietary["SEQN"].str.endswith(".0").any():

    raise ValueError(
        "Merged Dietary SEQN contains '.0'."
    )


print(
    "PASS: Final Dietary file contains "
    "9,762 unique participants."
)


# =================================================================================================
# 20. VERIFY NO DUPLICATE COLUMN NAMES
# =================================================================================================

duplicate_columns = (

    dietary.columns[
        dietary.columns.duplicated()
    ]

    .tolist()

)


if duplicate_columns:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_columns
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 21. VERIFY ENGINEERED DUPLICATES ARE ABSENT
# =================================================================================================

BAD_COLUMNS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE",

    "WTDRD1_X",

    "WTDRD1_Y",

    "WTDR2D_X",

    "WTDR2D_Y"

]


bad_found = [

    col

    for col in BAD_COLUMNS

    if col in dietary.columns

]


if bad_found:

    raise ValueError(
        "Engineered Dietary duplicate fields remain:\n"
        + str(
            bad_found
        )
    )


print(
    "PASS: No engineered shared-variable duplicates."
)


# =================================================================================================
# 22. CREATE FINAL LOGICAL COLUMN ORDER
# =================================================================================================

#
# Desired:
#
# SEQN
#
# shared fields
#
# all unique Day 1 fields
#
# all unique Day 2 fields
#

SHARED_ORDER = [

    "SEQN",

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


DAY1_UNIQUE_COLUMNS = [

    col

    for col in original_day1_columns

    if col not in SHARED_ORDER

]


DAY2_UNIQUE_COLUMNS = [

    col

    for col in original_day2_columns

    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )

]


FINAL_ORDER = (

    SHARED_ORDER

    +

    DAY1_UNIQUE_COLUMNS

    +

    DAY2_UNIQUE_COLUMNS

)


if len(FINAL_ORDER) != len(set(FINAL_ORDER)):

    raise ValueError(
        "Final column-order list contains duplicates."
    )


if set(FINAL_ORDER) != set(dietary.columns):

    missing = (
        set(dietary.columns)
        -
        set(FINAL_ORDER)
    )


    extra = (
        set(FINAL_ORDER)
        -
        set(dietary.columns)
    )


    raise ValueError(
        "Final ordering mismatch.\n\n"
        f"Missing:\n{sorted(missing)}\n\n"
        f"Extra:\n{sorted(extra)}"
    )


dietary = (
    dietary[
        FINAL_ORDER
    ]
)


print(
    "PASS: Final column order = "
    "shared -> Day 1 -> Day 2."
)


# =================================================================================================
# 23. DAY 1 EXACT TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DAY 1 SOURCE FIDELITY ---"
)


day1_check = (

    day1.merge(

        dietary[
            original_day1_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_values = (
        day1_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day1_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "Day 1 values changed during merge:\n"
        + str(
            day1_failures
        )
    )


print(
    "PASS: Every retained DR1TOT_E value "
    "preserved text-for-text."
)


# =================================================================================================
# 24. DAY 2 UNIQUE FIELD TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DAY 2 SOURCE FIDELITY ---"
)


day2_check_columns = (
    ["SEQN"]
    +
    DAY2_UNIQUE_COLUMNS
)


day2_check = (

    day2[
        day2_check_columns
    ]

    .merge(

        dietary[
            day2_check_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_values = (
        day2_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day2_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "Day 2 values changed during merge:\n"
        + str(
            day2_failures
        )
    )


print(
    "PASS: Every unique DR2TOT_E value "
    "preserved text-for-text."
)


# =================================================================================================
# 25. REPRESENTATIVE DIETARY DECIMAL CHECK
# =================================================================================================

#
# Diagnostic only.
#
# These variables MUST remain strings and MUST NOT be rounded.
#

DECIMAL_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TP204",

    "DR1_320Z",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TP204",

    "DR2_320Z"

]


print(
    "\n--- DIETARY DECIMAL PRESERVATION ---"
)


for col in DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated = int(
        dietary[col]
        .ne("")
        .sum()
    )


    decimal_tokens = int(

        (
            dietary[col].ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated:,}, "
        f"tokens containing decimal point="
        f"{decimal_tokens:,}"
    )


print(
    "PASS: No Dietary decimal field was "
    "rounded or integerized during merge."
)


# =================================================================================================
# 26. TINY-XPORT TOKEN CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_found = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        tiny_found[
            col
        ] = count


if tiny_found:

    raise ValueError(
        "Known XPORT tiny artifacts remain "
        "in corrected Dietary inputs:\n"
        + str(
            tiny_found
        )
    )


print(
    "PASS: No known tiny XPORT artifacts."
)


# =================================================================================================
# 27. SAVE FINAL CSV
# =================================================================================================

#
# IMPORTANT:
#
# No float_format=
# No rounding
#
# Since dataframe values remain strings, corrected decimal
# representations are written AS IS.
#

dietary.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 28. RE-READ SAVED CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 29. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed."
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column order changed."
    )


print(
    "PASS: Saved CSV structure preserved."
)


# =================================================================================================
# 30. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in dietary.columns:

    if not (
        dietary[col]
        .eq(
            final_df[col]
        )
        .all()
    ):

        post_save_failures.append(
            col
        )


if post_save_failures:

    raise ValueError(
        "CSV text changed during export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 31. FINAL SEQN CHECK
# =================================================================================================

if final_df["SEQN"].duplicated().any():

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if final_df["SEQN"].eq("").any():

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if final_df["SEQN"].str.endswith(".0").any():

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 32. FINAL ENGINEERED-DUPLICATE CHECK
# =================================================================================================

final_bad = [

    col

    for col in BAD_COLUMNS

    if col in final_df.columns

]


if final_bad:

    raise ValueError(
        "Final CSV still contains engineered duplicates:\n"
        + str(
            final_bad
        )
    )


print(
    "PASS: No redundant Day 2 shared-variable copies."
)


# =================================================================================================
# 33. FINAL SAMPLE WEIGHT CHECK AFTER SAVE
# =================================================================================================

final_wtdrd1 = int(
    final_df["WTDRD1"]
    .ne("")
    .sum()
)


final_wtdr2d = int(
    final_df["WTDR2D"]
    .ne("")
    .sum()
)


if final_wtdrd1 != 9255:

    raise ValueError(
        "Final WTDRD1 count differs from CDC."
    )


if final_wtdr2d != 7838:

    raise ValueError(
        "Final WTDR2D count differs from CDC."
    )


print(
    "PASS: Final sample weights match CDC:"
)

print(
    "  WTDRD1 = 9,255"
)

print(
    "  WTDR2D = 7,838"
)


# =================================================================================================
# 34. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2007-2008 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle: 2007-2008"
)


print(
    "Source files:"
)


print(
    "  DR1TOT_E_corrected.csv"
)


print(
    "  DR2TOT_E_corrected.csv"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    ""
)


print(
    "Official Dietary counts:"
)


print(
    "  Total participant rows = 9,762"
)


print(
    "  Day 1 weight records   = 9,255"
)


print(
    "  Day 2 weight records   = 7,838"
)


print(
    ""
)


print(
    "Shared variables retained once:"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Final organization:"
)


print(
    "  SEQN"
)


print(
    "  Shared Dietary fields"
)


print(
    "  Day 1 fields"
)


print(
    "  Day 2 fields"
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Engineered Dietary duplicates: 0"
)


print(
    "Known tiny XPORT artifacts: 0"
)


print(
    "Day 1 retained-value changes: 0"
)


print(
    "Day 2 retained-value changes: 0"
)


print(
    "Dietary decimal reformatting: 0"
)


print(
    "=" * 110
)

NHANES 2007-2008 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_E shape: (9762, 164)
DR2TOT_E shape: (9762, 83)
PASS: Both Dietary files contain 9,762 rows.

DR1TOT_E:
  Rows: 9,762
  Unique SEQN: 9,762
  Blank SEQN: 0
  Duplicate SEQN: 0
  SEQN ending '.0': 0
  Invalid SEQN formatting: 0

DR2TOT_E:
  Rows: 9,762
  Unique SEQN: 9,762
  Blank SEQN: 0
  Duplicate SEQN: 0
  SEQN ending '.0': 0
  Invalid SEQN formatting: 0

PASS: SEQN integrity validated.

--- PARTICIPANT SET CHECK ---
Only in DR1TOT_E: 0
Only in DR2TOT_E: 0
PASS: Both files contain the same 9,762 SEQN.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared variables found.

--- SHARED VARIABLE FIDELITY CHECK ---

WTDRD1
  Missing-position mismatches: 0
  Numeric mismatches: 0
  Text-format differences: 0

WTDR2D
  Missing-position mismatches: 0
  Numeric mismatches: 0
  Text-format differences: 0

DRABF
  Missing-position mismatches: 0
  Numeric mismatches: 0
  Text-format differ